# 01 — Acquisition des données TMDB

## Objectif

Ce notebook documente la phase d'**acquisition des données** du projet CinéVision. Nous collectons des données de films depuis l'API **TMDB (The Movie Database)** afin de constituer un dataset exploitable pour l'analyse exploratoire et la modélisation ML.

## Démarche

1. **Choix de la source** : API TMDB, gratuite, bien documentée, accès illimité avec rate limit raisonnable
2. **Stratégie de collecte** : films sortis entre 1980 et 2024 (45 ans), triés par popularité décroissante, avec au moins 50 votes (filtre qualité)
3. **Volume cible** : 200 pages × 20 films = ~4 000 films bruts
4. **Enrichissement** : pour chaque film, récupération des détails complets (budget, recettes, runtime) et des credits (casting + réalisateur)

## 1. Imports et configuration

In [1]:
import sys
from pathlib import Path

# Ajout du dossier src au path pour importer nos modules
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_acquisition import (
    get_api_key,
    discover_movies,
    enrich_movies,
    save_to_csv,
)

print(f"Pandas version : {pd.__version__}")
print(f"Project root : {PROJECT_ROOT}")

Pandas version : 2.2.3
Project root : C:\Users\lucie\OneDrive - Ynov\Suivi de projet fil rouge\cinevision


## 2. Authentification API

La clé API est chargée depuis le fichier `.env` à la racine du projet (jamais commité sur Git pour des raisons de sécurité).

In [2]:
api_key = get_api_key()
print(f"Clé API chargée : {api_key[:8]}...{api_key[-4:]} (masquée)")

Clé API chargée : 7e3a401c...25e3 (masquée)


## 3. Découverte des films

Premier appel à l'endpoint `/discover/movie` avec les paramètres suivants :
- **Période** : 1980 à 2024 (45 ans)
- **Tri** : popularité décroissante
- **Filtre qualité** : au moins 50 votes
- **Pages** : 200 (20 films par page → ~4 000 films)

### Pourquoi ces paramètres ?

- **1980-2024** : couvre l'ère du blockbuster moderne et permet une analyse temporelle sur 45 ans
- **min 50 votes** : élimine les films obscurs ou mal référencés (insuffisamment notés pour être fiables)
- **Tri popularité** : récupère les films les plus pertinents pour notre analyse

In [3]:
# NOTE : cette cellule est lourde (~5 min). Le résultat est sauvegardé en CSV.
# Si le fichier existe déjà, on saute cette étape.

raw_file = PROJECT_ROOT / 'data' / 'raw' / 'movies_raw.csv'

if raw_file.exists():
    print(f'Fichier existant détecté : {raw_file}')
    print('Chargement direct depuis le CSV...')
    df = pd.read_csv(raw_file)
else:
    print('Lancement de la collecte (durée ~30 min)...')
    movies = discover_movies(
        api_key,
        pages=200,
        start_year=1980,
        end_year=2024,
        min_votes=50,
    )
    enriched = enrich_movies(api_key, movies)
    save_to_csv(enriched, 'movies_raw.csv')
    df = pd.DataFrame(enriched)

print(f'\nDataset chargé : {df.shape[0]} films, {df.shape[1]} colonnes')

Fichier existant détecté : C:\Users\lucie\OneDrive - Ynov\Suivi de projet fil rouge\cinevision\data\raw\movies_raw.csv
Chargement direct depuis le CSV...

Dataset chargé : 4000 films, 23 colonnes


## 4. Aperçu du dataset brut

In [4]:
df.head()

,id,title,original_title,release_date,runtime,budget,revenue,vote_average,vote_count,popularity,...,production_companies,production_countries,belongs_to_collection,collection_name,director,composer,cast_top5,cast_top5_popularity,overview,tagline
0,10867,Malena,Malèna,2000-10-27,108,0,14493284,7.429,2521,197.6598,...,"['Medusa Film', 'Miramax', 'Pacific Pictures',...","['IT', 'US']",False,NaN,Giuseppe Tornatore,Ennio Morricone,"['Monica Bellucci', 'Giuseppe Sulfaro', 'Lucia...","[13.0921, 0.0, 0.2556, 0.2779, 0.2196]",12-year-old Renato experiences three significa...,"Too young to be a widow, too beautiful to be a..."
1,350,The Devil Wears Prada,The Devil Wears Prada,2006-06-29,109,35000000,326588371,7.397,13279,297.0386,...,"['Fox 2000 Pictures', 'Wendy Finerman Producti...",['US'],True,The Devil Wears Prada Collection,David Frankel,Theodore Shapiro,"['Meryl Streep', 'Anne Hathaway', 'Emily Blunt...","[8.0172, 27.5047, 12.9007, 7.2937, 5.3784]",A young woman from the Midwest gets more than ...,Meet Andy Sachs. A million girls would kill to...
2,318256,Hot Girls Wanted,Hot Girls Wanted,2015-01-24,84,0,0,6.086,717,202.0650,...,['Two to Tangle Productions'],['US'],False,NaN,Ronna Gradus,Daniel Ahearn,"['Stella May', 'Brian Omally', 'Ava Taylor', '...","[0.0, 0.0, 0.0, 0.0, 0.0793]",A first-ever look at the realities of the prof...,"Porn, the internet and the girl next door."
3,82023,Hotel Desire,Hotel Desire,2011-12-07,38,230758,0,6.200,145,57.5432,...,"['Von Fiessbach Film', 'teamWorx']",['DE'],False,NaN,Sergej Moya,Stefan Maria Schneider,"['Saralisa Volm', 'Clemens Schick', 'Jan-Grego...","[0.8216, 0.8887, 0.5408, 0.3893, 0.6445]",Antonia is a single mother who works as a hote...,NaN
4,440249,After Porn Ends 2,After Porn Ends 2,2017-03-28,90,500000,0,5.400,251,72.2206,...,"['Karbonshark', 'WeBros Entertainment']",['US'],True,After Porn Ends Collection,Bryce Wagoner,Simon H. Jay,"['Ginger Lynn', 'Lisa Ann', 'Janine Lindemulde...","[0.0, 0.0, 0.0, 0.0, 0.0]",After Porn Ends 2 picks up where its predecess...,NaN


In [5]:
print('Colonnes du dataset :')
for col in df.columns:
    print(f'  - {col} ({df[col].dtype})')

Colonnes du dataset :
  - id (int64)
  - title (object)
  - original_title (object)
  - release_date (object)
  - runtime (int64)
  - budget (int64)
  - revenue (int64)
  - vote_average (float64)
  - vote_count (int64)
  - popularity (float64)
  - original_language (object)
  - status (object)
  - genres (object)
  - production_companies (object)
  - production_countries (object)
  - belongs_to_collection (bool)
  - collection_name (object)
  - director (object)
  - composer (object)
  - cast_top5 (object)
  - cast_top5_popularity (object)
  - overview (object)
  - tagline (object)


In [6]:
print('Statistiques de qualité des données :')
print(f"  - Films avec budget renseigné  : {(df['budget'] > 0).sum()} / {len(df)} ({(df['budget'] > 0).mean()*100:.1f}%)")
print(f"  - Films avec recettes renseignées : {(df['revenue'] > 0).sum()} / {len(df)} ({(df['revenue'] > 0).mean()*100:.1f}%)")
print(f"  - Films avec réalisateur identifié : {df['director'].notna().sum()} / {len(df)} ({df['director'].notna().mean()*100:.1f}%)")
print(f"  - Films avec casting (top 5)      : {df['cast_top5'].apply(lambda x: len(eval(x) if isinstance(x, str) else x) > 0).sum()} / {len(df)}")

Statistiques de qualité des données :
  - Films avec budget renseigné  : 3124 / 4000 (78.1%)
  - Films avec recettes renseignées : 3372 / 4000 (84.3%)
  - Films avec réalisateur identifié : 4000 / 4000 (100.0%)
  - Films avec casting (top 5)      : 3998 / 4000


In [7]:
# Répartition temporelle des films collectés
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
year_counts = df['release_year'].value_counts().sort_index()
print('Films par décennie :')
decade = (year_counts.index // 10) * 10
by_decade = year_counts.groupby(decade).sum()
for d, count in by_decade.items():
    print(f"  {int(d)}s : {count} films")

Films par décennie :
  1980s : 290 films
  1990s : 564 films
  2000s : 983 films
  2010s : 1337 films
  2020s : 826 films


## 5. Conclusion de la phase d'acquisition

Le dataset brut est constitué et sauvegardé dans `data/raw/movies_raw.csv`. Il servira de base à la phase suivante : **nettoyage et feature engineering** (notebook `02_data_cleaning.ipynb`).

### Points clés

- ✅ Données récupérées depuis une source ouverte et fiable (TMDB)
- ✅ Pipeline de collecte modulaire et réutilisable (`src/data_acquisition.py`)
- ✅ Gestion du rate limiting et des erreurs API
- ✅ Volume de données suffisant pour l'analyse et le ML
- ⚠️ Une partie des films a des champs `budget` et `revenue` manquants → à gérer dans la phase de nettoyage